# M6 Synthetic Evaluation

Notebook for inspecting tuned synthetic evaluation results of the M6 scoring module.
The logic stays in `evaluation.py`; this notebook only visualizes comparisons and exported artifacts.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from backend.app.modules.m6_scoring.evaluation import (
    build_fixture_report,
    compare_models,
    evaluate_hybrid_model,
)

sns.set_theme(style='whitegrid')


In [ ]:
TRAIN_SAMPLES = 300
TEST_SAMPLES = 120
SEED = 42
RESULTS_DIR = Path('backend/tests/m6_scoring/results/latest')


In [ ]:
balanced_comparison = compare_models(
    train_sample_count=TRAIN_SAMPLES,
    test_sample_count=TEST_SAMPLES,
    seed=SEED,
    test_profile_mix='balanced',
)
balanced_comparison


In [ ]:
stress_comparison = compare_models(
    train_sample_count=TRAIN_SAMPLES,
    test_sample_count=TEST_SAMPLES,
    seed=SEED,
    test_profile_mix='stress',
)
stress_comparison


In [ ]:
fixture_report = build_fixture_report()
fixture_report


In [ ]:
gbr = evaluate_hybrid_model(train_sample_count=TRAIN_SAMPLES, test_sample_count=TEST_SAMPLES, seed=SEED, model_family='gbr')
gbr_predictions = gbr['predictions']
gbr['metrics']


In [ ]:
delta_path = RESULTS_DIR / 'vs_pre_tuning.csv'
if delta_path.exists():
    pd.read_csv(delta_path)
else:
    pd.DataFrame({'note': ['Run backend/tests/m6_scoring/run_evaluation.py first to materialize the delta file.']})


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(data=gbr_predictions, x='target_rpi', y='predicted_rpi', hue='confidence_band', ax=axes[0])
axes[0].set_title('GBR: Target vs Predicted RPI')

status_counts = gbr_predictions['predicted_status'].value_counts().reset_index()
status_counts.columns = ['status', 'count']
sns.barplot(data=status_counts, x='status', y='count', ax=axes[1])
axes[1].set_title('GBR: Status Distribution')
axes[1].tick_params(axis='x', rotation=25)

confidence_counts = gbr_predictions['confidence_band'].value_counts().reset_index()
confidence_counts.columns = ['band', 'count']
sns.barplot(data=confidence_counts, x='band', y='count', ax=axes[2])
axes[2].set_title('GBR: Confidence Bands')

plt.tight_layout()
